# Deep Learning sur N-Grammes — CNN 1D

CNN 1D PyTorch sur features N-grammes pour la classification de sentiment Yelp.

**Architecture** : Conv1d(k=3,4,5) parallèle → MaxPool → Dense → Output

**Deux tâches** : Polarité (3 classes) et Score (1-5 étoiles)

In [ ]:
import sys
sys.path.insert(0, '../..')

import os
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.constants import POLARITY_NAMES, SCORE_NAMES
from src.ml_utils import load_and_prepare, split_data
from src.dl_utils import get_device, set_seed, make_loaders, train_model, evaluate_model
from src.evaluation import plot_confusion, plot_training_curves, print_report
from src import setup_plot_style

setup_plot_style()
set_seed()
device = get_device()
MODELS_DIR = '../../models/'
os.makedirs(MODELS_DIR, exist_ok=True)
print(f'Device : {device}')

## 1. Chargement et Préparation

In [ ]:
df = load_and_prepare()
data = split_data(df['text'], df['polarity'], df['stars'])
print(f"Train: {len(data['X_train'])} | Val: {len(data['X_val'])} | Test: {len(data['X_test'])}")

## 2. Vectorisation N-Grammes

In [ ]:
VOCAB_SIZE = 10_000

vectorizer = CountVectorizer(max_features=VOCAB_SIZE, min_df=5, max_df=0.7, ngram_range=(1, 2))

X_train = vectorizer.fit_transform(data['X_train']).toarray()
X_val = vectorizer.transform(data['X_val']).toarray()
X_test = vectorizer.transform(data['X_test']).toarray()

# Score 0-indexed pour PyTorch
y_sc_train = (data['y_sc_train'] - 1).values.astype(int)
y_sc_val = (data['y_sc_val'] - 1).values.astype(int)
y_sc_test = (data['y_sc_test'] - 1).values.astype(int)

print(f'Matrice N-grammes : {X_train.shape}')

## 3. Architecture CNN 1D

Convolutions parallèles (Kim 2014) avec kernels 3, 4, 5 suivies de max-pooling global.

In [ ]:
class TextCNN1D(nn.Module):
    def __init__(self, vocab_size, num_classes=5, num_filters=128,
                 kernel_sizes=(3, 4, 5), dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(1, num_filters, k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(num_filters * len(kernel_sizes), 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, vocab) → (batch, 1, vocab)
        pooled = [self.relu(conv(x)).max(dim=2).values for conv in self.convs]
        out = torch.cat(pooled, dim=1)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        return self.fc2(out)

model_test = TextCNN1D(VOCAB_SIZE, 3)
print(model_test)
print(f'\nParamètres : {sum(p.numel() for p in model_test.parameters() if p.requires_grad):,}')
del model_test

## 4. Polarité (3 classes)

In [ ]:
train_pol, val_pol, test_pol = make_loaders(
    X_train, data['y_pol_train'].values,
    X_val, data['y_pol_val'].values,
    X_test, data['y_pol_test'].values,
    batch_size=64
)

model_pol = TextCNN1D(VOCAB_SIZE, num_classes=3)
model_pol, history_pol = train_model(
    model_pol, train_pol, val_pol, device=device, epochs=15, patience=3
)

In [ ]:
plot_training_curves(history_pol, 'CNN 1D N-Grammes — Polarité')

preds_pol, labels_pol = evaluate_model(model_pol, test_pol, device=device)
print_report(labels_pol, preds_pol, POLARITY_NAMES, 'CNN 1D — Polarité (Test)')
plot_confusion(labels_pol, preds_pol, POLARITY_NAMES, 'CNN 1D — Polarité')

## 5. Score (1-5 étoiles)

In [ ]:
train_sc, val_sc, test_sc = make_loaders(
    X_train, y_sc_train, X_val, y_sc_val, X_test, y_sc_test, batch_size=64
)

model_score = TextCNN1D(VOCAB_SIZE, num_classes=5)
model_score, history_score = train_model(
    model_score, train_sc, val_sc, device=device, epochs=15, patience=3
)

In [ ]:
plot_training_curves(history_score, 'CNN 1D N-Grammes — Score')

preds_sc, labels_sc = evaluate_model(model_score, test_sc, device=device)
print_report(labels_sc, preds_sc, SCORE_NAMES, 'CNN 1D — Score (Test)')
plot_confusion(labels_sc, preds_sc, SCORE_NAMES, 'CNN 1D — Score')

## 6. Sauvegarde

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

comparison = pd.DataFrame({
    'Tâche': ['Polarité', 'Score'],
    'Accuracy': [accuracy_score(labels_pol, preds_pol), accuracy_score(labels_sc, preds_sc)],
    'F1 Macro': [f1_score(labels_pol, preds_pol, average='macro'),
                 f1_score(labels_sc, preds_sc, average='macro')],
})
print("=== RÉSUMÉ CNN 1D N-GRAMMES ===")
display(comparison)

torch.save(model_pol.state_dict(), os.path.join(MODELS_DIR, 'cnn1d_ngram_polarity.pt'))
torch.save(model_score.state_dict(), os.path.join(MODELS_DIR, 'cnn1d_ngram_score.pt'))
joblib.dump(vectorizer, os.path.join(MODELS_DIR, 'count_vectorizer_deep.pkl'))
print(f"\nModèles sauvegardés dans {MODELS_DIR}")